# Pretrain Llama 3 Mini on TinyShakespeare (Colab)
This notebook pretrains a tiny Llama 3 model on the TinyShakespeare dataset.

**Architecture:**
- Hidden dim: 512 (vs 4096 in full Llama 3)
- Layers: 8 (vs 32)
- Heads: 8 (vs 32)
- Total params: ~40M (vs 8B)
- VRAM: ~4-6GB (fits Colab T4)

## Setup & Imports

In [ ]:
import os
import sys
import json
import time
import numpy as np
from pathlib import Path
from typing import List, Optional, Tuple
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Download TinyShakespeare

In [ ]:
def download_tinystakespeare():
    """Download TinyShakespeare dataset"""
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    path = Path("tiny_shakespeare.txt")

    if not path.exists():
        print("Downloading TinyShakespeare...")
        urllib.request.urlretrieve(url, path)
        print(f"Downloaded to {path}")

    with open(path, 'r') as f:
        text = f.read()

    print(f"Dataset size: {len(text):,} characters")
    print(f"Unique characters: {len(set(text))}")
    return text

text = download_tinystakespeare()
print(f"First 200 chars:\n{text[:200]}")

## Simple Character-Level Tokenizer

In [ ]:
class CharTokenizer:
    """Simple character-level tokenizer"""
    def __init__(self, text: str):
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        self.char_to_idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(self.chars)}

    def encode(self, text: str) -> List[int]:
        return [self.char_to_idx[ch] for ch in text]

    def decode(self, tokens: List[int]) -> str:
        return ''.join([self.idx_to_char[t] for t in tokens])

tokenizer = CharTokenizer(text)
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Characters: {tokenizer.chars[:20]}")

# Test tokenizer
sample = "Hello, World!"
encoded = tokenizer.encode(sample)
decoded = tokenizer.decode(encoded)
print(f"\nTest: '{sample}'")
print(f"Encoded: {encoded}")
print(f"Decoded: '{decoded}'")

## Dataset Class

In [ ]:
class TinyShakespeareDataset(Dataset):
    def __init__(self, text: str, tokenizer: CharTokenizer, seq_len: int = 128):
        self.tokens = tokenizer.encode(text)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.tokens[idx:idx + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.tokens[idx + 1:idx + self.seq_len + 1], dtype=torch.long)
        return x, y

# Create train/val split
split_idx = int(0.9 * len(text))
train_text = text[:split_idx]
val_text = text[split_idx:]

train_dataset = TinyShakespeareDataset(train_text, tokenizer, seq_len=128)
val_dataset = TinyShakespeareDataset(val_text, tokenizer, seq_len=128)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

# Test dataloader
x_batch, y_batch = next(iter(train_loader))
print(f"\nBatch shape: x={x_batch.shape}, y={y_batch.shape}")

## Llama 3 Mini Architecture

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization"""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """Rotary positional embeddings (RoPE)"""
    def __init__(self, dim: int, max_seq_len: int = 2048):
        super().__init__()
        self.dim = dim
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

        t = torch.arange(max_seq_len).type_as(inv_freq)
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, q, k):
        seq_len = q.shape[2]
        cos = self.cos_cached[:, :, :seq_len, :]
        sin = self.sin_cached[:, :, :seq_len, :]

        q_rot = (q * cos) + (self._rotate_half(q) * sin)
        k_rot = (k * cos) + (self._rotate_half(k) * sin)
        return q_rot, k_rot

    @staticmethod
    def _rotate_half(x):
        x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

In [ ]:
class Attention(nn.Module):
    """Multi-head attention with RoPE"""
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        assert dim % n_heads == 0
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.wq = nn.Linear(dim, dim, bias=False)
        self.wk = nn.Linear(dim, dim, bias=False)
        self.wv = nn.Linear(dim, dim, bias=False)
        self.wo = nn.Linear(dim, dim, bias=False)

        self.rope = RotaryPositionalEmbedding(self.head_dim)

    def forward(self, x, mask=None):
        bsz, seq_len, _ = x.shape

        q = self.wq(x).view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        q, k = self.rope(q, k)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores + mask

        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, v)
        output = output.transpose(1, 2).contiguous().view(bsz, seq_len, self.dim)

        return self.wo(output)

In [ ]:
class FeedForward(nn.Module):
    """SwiGLU feed-forward network"""
    def __init__(self, dim: int):
        super().__init__()
        hidden_dim = int(4 * dim)
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [ ]:
class TransformerBlock(nn.Module):
    """Transformer block"""
    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.attention = Attention(dim, n_heads)
        self.feed_forward = FeedForward(dim)
        self.attention_norm = RMSNorm(dim)
        self.ffn_norm = RMSNorm(dim)

    def forward(self, x, mask=None):
        h = x + self.attention(self.attention_norm(x), mask=mask)
        out = h + self.feed_forward(self.ffn_norm(h))
        return out

In [ ]:
class LlamaMini(nn.Module):
    """Llama 3 Mini - downscaled for pretraining"""
    def __init__(
        self,
        vocab_size: int,
        dim: int = 512,
        n_layers: int = 8,
        n_heads: int = 8,
        max_seq_len: int = 2048,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim

        self.embedding = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([
            TransformerBlock(dim, n_heads) for _ in range(n_layers)
        ])
        self.norm = RMSNorm(dim)
        self.output_proj = nn.Linear(dim, vocab_size, bias=False)

        # Causal mask
        self.register_buffer(
            "causal_mask",
            torch.triu(torch.full((max_seq_len, max_seq_len), float("-inf")), diagonal=1)
        )

    def forward(self, x):
        seq_len = x.shape[1]
        x = self.embedding(x)

        mask = self.causal_mask[:seq_len, :seq_len].unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, mask=mask)

        x = self.norm(x)
        logits = self.output_proj(x)

        return logits

In [ ]:
# Initialize model
model = LlamaMini(
    vocab_size=tokenizer.vocab_size,
    dim=512,
    n_layers=8,
    n_heads=8,
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Model size: {total_params * 4 / 1e9:.2f} GB (FP32)")

# Test forward pass
x_test = next(iter(train_loader))[0].to(device)
with torch.no_grad():
    logits = model(x_test)
print(f"Input shape: {x_test.shape}")
print(f"Logits shape: {logits.shape}")

## Training Loop

In [ ]:
def train_epoch(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0
    num_batches = 0

    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, model.vocab_size), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        
        # Print progress every 5 batches
        if (i + 1) % 5 == 0:
            print(f"  Batch {i+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    return total_loss / num_batches

def eval_epoch(model, val_loader, device):
    model.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, model.vocab_size), y.view(-1))

            total_loss += loss.item()
            num_batches += 1

    return total_loss / num_batches

## Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
num_epochs = 5

print(f"Training for {num_epochs} epochs...")
print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")
print()

for epoch in range(num_epochs):
    start_time = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, device)
    val_loss = eval_epoch(model, val_loader, device)

    elapsed = time.time() - start_time

    print(f"\nEpoch {epoch+1}/{num_epochs} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Time: {elapsed:.1f}s")

## Generation

In [ ]:
@torch.no_grad()
def generate(
    model,
    tokenizer,
    prompt: str,
    max_len: int = 100,
    temperature: float = 0.8,
    device: str = "cpu",
):
    """Generate text from prompt"""
    model.eval()

    tokens = tokenizer.encode(prompt)
    tokens = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_len):
        logits = model(tokens)
        logits = logits[:, -1, :] / temperature

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        tokens = torch.cat([tokens, next_token], dim=1)

        if next_token.item() == tokenizer.vocab_size - 1:
            break

    generated = tokenizer.decode(tokens[0].cpu().tolist())
    return generated

In [ ]:
# Generate samples
print("=" * 50)
print("Generated text:")
print("=" * 50)

prompts = [
    "ROMEO:",
    "The sun",
    "To be",
]

for prompt in prompts:
    text = generate(model, tokenizer, prompt, max_len=100, device=device)
    print(f"\nPrompt: '{prompt}'")
    print(f"Generated:\n{text}")
    print("-" * 50)

## Save Model

In [ ]:
# Save model checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'tokenizer_vocab': tokenizer.chars,
    'model_config': {
        'vocab_size': tokenizer.vocab_size,
        'dim': 512,
        'n_layers': 8,
        'n_heads': 8,
    }
}

torch.save(checkpoint, 'llama_mini_pretrained.pt')
print("Model saved to llama_mini_pretrained.pt")

## Load & Test

In [ ]:
def load_model(checkpoint_path: str, device: str = "cpu"):
    """Load model from checkpoint"""
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Recreate tokenizer
    tokenizer = CharTokenizer(''.join(checkpoint['tokenizer_vocab']))

    # Recreate model
    config = checkpoint['model_config']
    model = LlamaMini(**config)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)

    return model, tokenizer

# Load and test
loaded_model, loaded_tokenizer = load_model('llama_mini_pretrained.pt', device=device)
print("Model loaded successfully!")

# Test generation
text = generate(loaded_model, loaded_tokenizer, "JULIET:", max_len=150, device=device)
print(f"Generated:\n{text}")